#### 1. Combine csv files into one dataframe

In [ ]:
import glob
import pandas as pandas
from sklearn.model_selection import train_test_split

csv_folder = "../data/CICIOT2023/*.csv"
file_paths = glob.glob(csv_folder)

print(f"Found {len(file_paths)} files")

dataframe_list = [pandas.read_csv(file) for file in file_paths]
dataframe_combined = pandas.concat(dataframe_list, ignore_index=True)
dataframe_combined = dataframe_combined.reset_index(drop=True)

#### 2. Drop rows with null values

In [ ]:
dataframe_combined.dropna()

In [ ]:
dataframe_combined.shape

In [ ]:
dataframe_combined['label'].value_counts()

#### 3. Create smaller dataframe

In [21]:
def sample_or_take_all(group):
    sample_size = min(len(group), 1000)
    return group.sample(n=sample_size, random_state=42)

dataframe_sampled = dataframe_combined.groupby('label').apply(sample_or_take_all)
dataframe_sampled = dataframe_sampled.reset_index(level=0)

In [22]:
dataframe_sampled['label'].value_counts()

label
BenignTraffic              1000
DDoS-ACK_Fragmentation     1000
DDoS-HTTP_Flood            1000
DDoS-ICMP_Flood            1000
DDoS-ICMP_Fragmentation    1000
DDoS-PSHACK_Flood          1000
DDoS-RSTFINFlood           1000
DDoS-SYN_Flood             1000
DDoS-SlowLoris             1000
DDoS-SynonymousIP_Flood    1000
DDoS-TCP_Flood             1000
DDoS-UDP_Flood             1000
DDoS-UDP_Fragmentation     1000
DNS_Spoofing               1000
DictionaryBruteForce       1000
DoS-HTTP_Flood             1000
DoS-SYN_Flood              1000
DoS-TCP_Flood              1000
DoS-UDP_Flood              1000
MITM-ArpSpoofing           1000
Mirai-greeth_flood         1000
Mirai-greip_flood          1000
Mirai-udpplain             1000
Recon-HostDiscovery        1000
Recon-OSScan               1000
Recon-PortScan             1000
VulnerabilityScan          1000
BrowserHijacking            957
SqlInjection                886
CommandInjection            871
XSS                         608
Ba

#### 4. Separate target from features

In [23]:
import numpy as numpy

targets = dataframe_sampled['label']
features = dataframe_sampled.select_dtypes(include=[numpy.number])

In [ ]:
features.shape

#### 5. K Fold validation

In [24]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

model = make_pipeline(
    StandardScaler(),
    SVC(kernel='rbf', random_state=0, gamma='scale', C=1.0)
)

k_fold = KFold(n_splits=5, shuffle=True, random_state=0)

cv_results = cross_val_score(model, features, targets, cv=k_fold, scoring='accuracy', n_jobs=-1)

print(cv_results)
cv_results.mean()

[0.61929741 0.61653418 0.62734499 0.61891892 0.61923688]


np.float64(0.6202664770299207)

#### A. Split training and testing

In [ ]:
from sklearn.model_selection import train_test_split

features_train, features_test, targets_train, targets_test = train_test_split(features, targets, test_size=0.2, random_state=42)

#### B. Apply Scaler

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
features_train_scaled = scaler.fit_transform(features_train)
features_test_scaled = scaler.transform(features_test)

#### C. Train Model

In [ ]:
from sklearn.svm import SVC

svc = SVC(kernel='rbf', random_state=0, gamma=1, C=1)
model2 = svc.fit(features_train_scaled, targets_train)